# SO4/T4 and Calvin Site 1 branching review

Run all cells to read the recorded comparisons, source crosswalk and current settings.
The saved simulations use the frozen Python 3.10 source in this project.
Set `RUN_PANELS = True` to execute the comparison commands. They resume from
source-checked trials in the same folder and preserve the historical receipts.

SO4 is ranked by joint 3D allocation of below-ground length.
Calvin uses dry mass and RLD at paired core positions and depths.

In [ ]:
from pathlib import Path
from hashlib import sha256
import json
import os
import subprocess

import pandas as pd
from IPython.display import display, Image

task_name = '.runtime/schema4-issue-closure-20261119'
cwd = Path.cwd().resolve()
project = next(p for p in (cwd, *cwd.parents) if (p / task_name / 'comparison/protocol.json').is_file())
task = project / task_name
results = task / 'comparison'
read_json = lambda path: json.loads(path.read_text())
protocol = read_json(results / 'protocol.json')
for path, digest in protocol['inputs_sha256'].items():
    assert sha256((project / path).read_bytes()).hexdigest() == digest, path
for cases in protocol['configs'].values():
    for case, digest in cases.items():
        path = results / 'configs' / f'{case}.schema4.private.json'
        assert sha256(path.read_bytes()).hexdigest() == digest, path
print('Frozen model source', protocol['source_sha256'])

In [ ]:
RUN_PANELS = False
if RUN_PANELS:
    source = task / 'source'
    python = project / '.runtime/envs/py310-qualification/bin/python'
    environment = dict(os.environ, PYTHONPATH=str(source), PYTHONDONTWRITEBYTECODE='1',
                       OMP_NUM_THREADS='1', OPENBLAS_NUM_THREADS='1', MKL_NUM_THREADS='1')
    subprocess.run([str(python), '../compare.py', 'so4'], cwd=source, env=environment, check=True)
    subprocess.run([str(python), '../compare_calvin.py'], cwd=source, env=environment, check=True)

## SO4/T4 at 18 months

The three-dimensional TV sum compares allocations on the same fixed soil frame.
The established scalar and marginal criteria are shown beside the spatial objective.
The earlier long-panel result remains in the source bundle.

In [ ]:
so4 = pd.read_csv(results / 'so4_development_scores.csv')
display(so4[['candidate', 'voxel_cm', 'joint_xyz_tv', 'across_depth_tv',
             'along_depth_tv', 'max_scalar_rel_error', 'max_historical_marginal_tv']]
        .sort_values(['voxel_cm', 'joint_xyz_tv']).round(3))
print(read_json(results / 'so4_selection.json'))
display(Image(filename=str(task / 'so4_allocation_projections.png')))

## Calvin Site 1 and the mature structural candidates

We held the 28 crown axes, length and taper laws, tissue density, observation
support and 0.93 mm attachment rule fixed. The first two parent-order laws use
the mature comparison values. Recovery multipliers affect the conditional
daughter-count distribution of order-2 parents. The sparse-order-3 case
changes the third parent-order law separately.

In [ ]:
fit = pd.read_csv(results / 'calvin_development_scores.csv')
held = pd.read_csv(results / 'calvin_confirmation_scores.csv')
columns = ['candidate', 'split', 'spatial_score', 'rld_nmae', 'mass_nmae',
           'rld_joint_tv', 'mass_joint_tv', 'rld_depth_tv', 'mass_depth_tv']
display(fit[columns].sort_values('spatial_score').round(3))
display(held[columns].sort_values(['split', 'spatial_score']).round(3))
display(Image(filename=str(task / 'calvin_held_profiles.png')))

In [ ]:
cases = read_json(task / 'public_axis_reconciliation.json')
display(pd.DataFrame([{
    'vine': row['vine'], 'duplicate_policy': row['duplicate_policy'],
    'raw': row['raw_counts'], 'RSML': row['rsml_counts'],
    'matched_axes': len(row['matches']), 'raw_only': len(row['raw_only']),
    'eligible_for_geometry_summary': row['included_in_geometric_summary']
} for row in cases if 'matches' in row]))
first = pd.read_csv(task / 'first_site_and_probability.csv')
display(first[first.case.isin(['calvin_qc', 'mature_x1', 'mature_x2', 'mature_x3',
                              'starter', 'SO4_T4_local111_review'])].round(4))